# 02 — Fusion ISCO 4 chiffres (add-on SHP) × indice OIT (ILO)

**Règle :** on n'affiche jamais de lignes individuelles du SHP (pas de `head()`, pas de `display(df)`).
Le contrat FORS interdit de rendre les données accessibles à des tiers, y compris des outils d'IA.
On n'affiche que des informations agrégées : `info()`, `shape`, `value_counts()`, moyennes.

**Avant de lancer :**
- avoir son propre accès à l'add-on FORS (réf. 3260) et dézipper `Data_with_additional_contract.zip` dans son dossier ;
- avoir lancé `01_shp_cleaning.py` avec la sortie dans `Data/private/shp_population_income.parquet`.

In [1]:
import pandas as pd
import numpy as np

## 1. Chemins
Chemins relatifs depuis le dossier `Notebooks/`, comme dans les séries (`./data/...`).

In [2]:
ADDON_PATH = "../Data/raw/SHP/swissubase_3260_1_1/Data_with_additional_contract/SHP_EXT_ISCO4_OCCUPA_USER.dta"
SHP_PATH   = "../Data/private/shp_population_income.parquet"   # fichier produit par 01_shp_cleaning
ILO_PATH   = "../Data/processed/ILO_exposure_by_occupation_ISCO4.csv"
OUT_PATH   = "../Data/private/shp_isco4_ilo.parquet"

## 2. Découvrir l'add-on (aucune ligne affichée)
Le fichier est en **format large** : une ligne par personne, une colonne par vague.
`IS4MAJyy` = code ISCO-08 à 4 chiffres de l'**emploi principal** pour l'année 20yy.

In [3]:
addon = pd.read_stata(ADDON_PATH, convert_categoricals=False)
print(addon.shape)
addon.info()

(50250, 121)
<class 'pandas.DataFrame'>
RangeIndex: 50250 entries, 0 to 50249
Columns: 121 entries, idpers to XIS4LA__
dtypes: int16(62), int32(57), int8(2)
memory usage: 17.0 MB


## 3. Passer en format long (`melt`, Week 2)
Une ligne par personne **et** par année, comme le fichier de cleaning.

In [4]:
cols_isco4 = ["IS4MAJ16", "IS4MAJ17", "IS4MAJ18", "IS4MAJ19", "IS4MAJ20",
              "IS4MAJ21", "IS4MAJ22", "IS4MAJ23", "IS4MAJ24"]

addon = addon.melt(id_vars="idpers", value_vars=cols_isco4,
                   var_name="variable", value_name="is4maj")

# L'année vient des 2 derniers caractères du nom de colonne
addon["year"] = 2000 + addon["variable"].str[-2:].astype(int)
addon = addon.drop("variable", axis=1)

print(addon.shape)
print(addon["year"].value_counts().sort_index())

(452250, 3)
year
2016    50250
2017    50250
2018    50250
2019    50250
2020    50250
2021    50250
2022    50250
2023    50250
2024    50250
Name: count, dtype: int64


## 4. Nettoyer le code ISCO 4
Les **non-réponses sont codées par des nombres négatifs** (-1, -2, -3, -8) → `NaN` (Week 2, valeurs manquantes codées).
Le code est transformé en **texte à 4 caractères** (`zfill`) pour garder le zéro initial, comme `isco3` dans le cleaning.

In [5]:
addon.loc[addon["is4maj"] < 0, "is4maj"] = np.nan
addon["isco4"] = addon["is4maj"].apply(lambda x: str(int(x)).zfill(4) if pd.notna(x) else np.nan)
addon = addon[["idpers", "year", "isco4"]]

print("Doublons idpers + year :", addon.duplicated(subset=["idpers", "year"]).sum())
print("Part de ISCO 4 manquant :", round(addon["isco4"].isna().mean(), 3))
print("Nombre de codes différents :", addon["isco4"].nunique())

Doublons idpers + year : 0
Part de ISCO 4 manquant : 0.863
Nombre de codes différents : 421


## 5. Fusion avec le fichier de cleaning
`how="left"` garde toutes les lignes du fichier de cleaning ; `indicator=True` (Week 4) montre si chaque ligne a trouvé un code.

In [6]:
shp = pd.read_parquet(SHP_PATH)
print("Lignes avant fusion :", shp.shape[0])

df = pd.merge(shp, addon, on=["idpers", "year"], how="left", indicator=True)
print("Lignes après fusion :", df.shape[0])   # doit être identique
print(df["_merge"].value_counts())
df = df.drop("_merge", axis=1)

# Contrôle : les 3 premiers chiffres de isco4 doivent correspondre à isco3
both = df[df["isco4"].notna() & df["isco3"].notna()]
print("Part cohérente :", round((both["isco4"].str[:3] == both["isco3"]).mean(), 3))

Lignes avant fusion : 40796
Lignes après fusion : 40796
_merge
both          40796
left_only         0
right_only        0
Name: count, dtype: int64
Part cohérente : 1.0


## 6. Fusion avec l'indice OIT à 4 chiffres

In [7]:
ilo = pd.read_csv(ILO_PATH)
ilo["isco4"] = ilo["ISCO_08"].astype(str).str.zfill(4)

ilo4 = ilo[["isco4", "exposure_mean_2025", "exposure_category_2025"]]
ilo4 = ilo4.rename(columns={"exposure_mean_2025": "exposure4",
                            "exposure_category_2025": "exposure4_cat"})
print("Doublons dans l'OIT :", ilo4.duplicated(subset=["isco4"]).sum())   # doit être 0

df = pd.merge(df, ilo4, on="isco4", how="left", indicator=True)
print("Lignes après fusion :", df.shape[0])
print(df["_merge"].value_counts())

# Codes du SHP sans score (codes et nombre d'observations seulement)
sans_score = df[(df["isco4"].notna()) & (df["_merge"] == "left_only")]
print(sans_score["isco4"].value_counts().head(20))
df = df.drop("_merge", axis=1)

Doublons dans l'OIT : 0
Lignes après fusion : 40796
_merge
both          36860
left_only      3936
right_only        0
Name: count, dtype: int64
isco4
3000    900
2000    501
2500    244
1210    230
4320    225
2140    215
7200    201
2210    174
2610    162
2510    150
2300    130
3100    128
9000     93
8100     89
1000     81
1320     47
8300     43
8210     38
3435     33
1340     30
Name: count, dtype: int64


## 7. Codes avec des zéros : score du groupe connu
Quand la réponse est trop vague, le SHP complète le code avec des zéros (`2500` = « informaticien sans précision »).
Ces codes n'existent pas dans l'OIT. On leur donne la **moyenne OIT du groupe connu** (`2500` → moyenne des `25xx`),
et `exposure_source` indique d'où vient le score (test de robustesse possible sans les lignes `groupe`).

In [8]:
# Moyennes OIT à 1, 2 et 3 chiffres
moyennes = []
for k in [1, 2, 3]:
    m = ilo.groupby(ilo["isco4"].str[:k])["exposure_mean_2025"].mean()
    moyennes.append(m)
moyennes = pd.concat(moyennes)   # index = "3", "25", "214", ...

# Partie connue du code : on enlève les zéros de fin (2500 -> "25")
df["isco_connu"] = df["isco4"].str.rstrip("0")

df["exposure_main"] = df["exposure4"].fillna(df["isco_connu"].map(moyennes))
df["exposure_source"] = np.where(df["exposure4"].notna(), "isco4", "groupe")
df.loc[df["exposure_main"].isna(), "exposure_source"] = "aucun"

print(df["exposure_source"].value_counts())

exposure_source
isco4     36860
groupe     3856
aucun        80
Name: count, dtype: int64


## 8. Sauvegarde (dossier privé, jamais sur GitHub)

In [9]:
df.to_parquet(OUT_PATH, index=False)
print("Sauvegardé :", OUT_PATH, df.shape)

Sauvegardé : ../Data/private/shp_isco4_ilo.parquet (40796, 36)
